# 01 — Create Heatmap

`POST /v1/heatmap` generates a thermal map (GeoJSON tile layer + statistics) over a polygon AOI.

**Plan:** available on both Basic (≤10 mi²) and Premium (≤50 mi²).

Inputs:
- `polygon_aoi` — GeoJSON FeatureCollection (coordinates are `[lon, lat]`)
- `date_time` — `start_date`, `filter_type` (1=single hour, 2=range, 3=single day), and matching `start_time`/`end_time`
- `granularity` — 60, 80, or 100 meters

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from dotenv import load_dotenv; load_dotenv(pathlib.Path.cwd().parent / '.env')

from fortyguard import FortyGuardClient
from fortyguard.samples import MANHATTAN_POLYGON

client = FortyGuardClient()

In [ ]:
response = client.create_heatmap(
    polygon_aoi=MANHATTAN_POLYGON,
    start_date='2024-07-15',
    start_time='14:00',
    filter_type=1,     # single hour
    granularity=100,
)

activity_id = response['activity_id']
result = response['result']
print(f'activity_id: {activity_id}')
print(f'result keys: {list(result.keys())}')

In [ ]:
# Look at the aggregated statistics.
stats = result.get('stats_data', {})
temp_stats = stats.get('Temperature_stats') or stats.get('temperature_stats') or {}
print('Temperature stats:')
for key, value in temp_stats.items():
    print(f'  {key:>20}: {value}')

In [ ]:
# Plot the temperature distribution if present.
import matplotlib.pyplot as plt

dist = stats.get('Overall_temperature_distribution') or stats.get('overall_temperature_distribution')
if dist:
    plt.figure(figsize=(8, 3))
    plt.hist(dist, bins=40, color='tomato', edgecolor='white')
    plt.xlabel('Temperature (°C)'); plt.ylabel('Tile count')
    plt.title('Heatmap tile temperature distribution'); plt.tight_layout(); plt.show()
else:
    print('No distribution data returned — inspect `stats` above for alternate fields.')

In [ ]:
# Visualise the GeoJSON tiles on a Folium map, coloured by temperature.
import folium

map_data = result.get('map_data')
if map_data and map_data.get('features'):
    temps = [f['properties'].get('temperature') for f in map_data['features'] if 'temperature' in f.get('properties', {})]
    lo, hi = (min(temps), max(temps)) if temps else (0, 1)
    
    def _style(feature):
        t = feature['properties'].get('temperature', lo)
        frac = 0 if hi == lo else (t - lo) / (hi - lo)
        r = int(255 * frac); b = int(255 * (1 - frac))
        return {'fillColor': f'#{r:02x}00{b:02x}', 'color': '#00000000', 'fillOpacity': 0.65, 'weight': 0}
    
    centroid = MANHATTAN_POLYGON['features'][0]['geometry']['coordinates'][0][0]
    fmap = folium.Map(location=[centroid[1], centroid[0]], zoom_start=14, tiles='cartodbpositron')
    folium.GeoJson(map_data, style_function=_style).add_to(fmap)
    fmap
else:
    print('No map_data features present — result shape may differ for this request.')